**Autor:** Alan Sastre  
**GitHub:** [github.com/alansastre/genai-fundamentos](https://github.com/alansastre/genai-fundamentos)

---

## Las limitaciones clave

| Limitación | Qué significa | Impacto práctico |
|------------|---------------|------------------|
| **Alucinaciones** | Inventa información con confianza | Datos falsos presentados como hechos |
| **Conocimiento desactualizado** | Solo sabe lo que vio en entrenamiento | Respuestas obsoletas |
| **Ventana de contexto** | Olvida en conversaciones largas | Contradicciones y pérdida de coherencia |
| **Razonamiento limitado** | Falla en cálculos y lógica | No fiable para matemáticas |
| **Sesgos** | Refleja prejuicios de los datos | Respuestas parciales |
| **Riesgos de seguridad** | Vulnerable a manipulación | Puede filtrar datos o ignorar restricciones |

> Ninguna de estas limitaciones se ha eliminado por completo. La clave es conocerlas y aplicar mitigaciones.

## Por qué ocurren: fundamentos

Un LLM no *sabe* cosas en el sentido tradicional. Ha aprendido **patrones estadísticos** de correlación entre tokens durante el entrenamiento. Cuando genera texto:

1. **Predice el siguiente token** más probable según lo que ha visto
2. **No verifica hechos** contra ninguna base de datos
3. **No tiene noción de certeza** sobre lo que dice
4. **Está optimizado para responder**, no para decir "no sé"

```mermaid
%%{init: {'theme': 'default'}}%%
flowchart LR
    P["Prompt"] --> M["LLM"]
    M --> PRED["Predicción estadística"]
    PRED --> R["Respuesta"]

    DB[("Verificación de hechos")]
    M -.-x DB

    style DB fill:#ff6b6b,color:#fff
    style M fill:#9f7aea,color:#fff
```

> El modelo genera texto que *parece correcto* basándose en patrones, independientemente de su veracidad. Esta es la raíz de todas las limitaciones.

## Alucinaciones

Las **alucinaciones** son respuestas que contienen información incorrecta presentada con total confianza. Ejemplos típicos:

- **Hechos inventados**: fechas o eventos que nunca ocurrieron
- **Citas fabricadas**: referencias bibliográficas inexistentes
- **Datos falsos**: estadísticas inventadas pero plausibles
- **Atribuciones erróneas**: frases asignadas a quien no las dijo

**Causas principales:**
- Aprendió de internet, que contiene errores y desinformación
- Extrapola patrones más allá de lo que los datos soportan
- No tiene indicador interno de certeza sobre lo que dice
- Prefiere generar algo plausible antes que admitir "no lo sé"

> Las alucinaciones son peligrosas porque el modelo las presenta igual que la información correcta. No hay señales visuales ni indicadores de confianza que las distingan.

## Limitaciones técnicas

### Conocimiento desactualizado

El modelo tiene una **fecha de corte**: solo conoce lo que existía cuando se entrenó. No puede aprender de conversaciones ni actualizarse espontáneamente.

**Consecuencias:**
- Eventos posteriores al entrenamiento son desconocidos
- Datos dinámicos (precios, leyes, versiones de software) pueden estar obsoletos
- Puede inventar información plausible pero incorrecta sobre temas recientes

### Ventana de contexto

Los LLM tienen un **límite máximo de tokens** que pueden procesar simultáneamente. Este límite representa el número de vectores de embeddings que el modelo maneja a la vez, ya que el mecanismo de atención compara cada token con todos los demás.

**Por qué importa:**

| Problema | Causa técnica |
|----------|---------------|
| **Dilución de atención** | Con contextos muy largos, la atención se distribuye entre demasiados tokens, reduciendo la precisión |
| **Entrenamiento no uniforme** | Los modelos no fueron entrenados uniformemente para usar todo el contexto disponible |
| **Pérdida de posicionamiento** | Los embeddings de posición pierden precisión en los extremos de la ventana |
| **Dificultad para priorizar** | Con miles de tokens, identificar qué información es relevante se vuelve más difícil |

**Efectos prácticos:**
- En conversaciones largas, "olvida" el contexto inicial
- Documentos extensos no caben completos, requiriendo estrategias de fragmentación
- Puede contradecirse entre el principio y el final de una respuesta larga
- El rendimiento se degrada cerca del límite máximo de tokens

> Aunque los modelos modernos anuncian ventanas de 200K o incluso 1M de tokens, usarlas cerca del límite no garantiza la misma calidad que con contextos más reducidos.

### Razonamiento limitado

Los LLM predicen la *forma* de una respuesta correcta sin ejecutar cálculos reales:

| Tarea | Limitación |
|-------|------------|
| Matemáticas | Errores en operaciones, especialmente con números grandes |
| Lógica multi-paso | Pierde el hilo en cadenas de razonamiento complejas |
| Conteo | Puede equivocarse contando elementos |
| Planificación | Dificultad para anticipar consecuencias de acciones |

> Los modelos de razonamiento han mejorado considerablemente en estas tareas, pero a costa de mayor latencia y coste.

### Sesgos heredados

Los LLM reflejan los prejuicios presentes en sus datos de entrenamiento:
- Estereotipos de género, raza, nacionalidad
- Perspectivas culturales dominantes (generalmente anglosajonas)
- Opiniones presentadas como hechos objetivos
- Subrepresentación de idiomas y culturas minoritarias

## Mitigación práctica

### Verificación y validación

La estrategia más directa: **no confiar ciegamente**.

- Verificar datos factuales críticos con fuentes externas
- Revisar respuestas importantes con expertos del dominio
- Establecer procesos de validación antes de usar la información

### Prompting efectivo

Mejorar la calidad de las respuestas desde el prompt:

```mermaid
%%{init: {'theme': 'default'}}%%
flowchart LR
    PP["Prompt vago"] --> RP["Respuesta imprecisa"]
    PB["Prompt estructurado"] --> RB["Respuesta fiable"]

    style RP fill:#ff6b6b,color:#fff
    style RB fill:#4ade80,color:#fff
```

- Proporcionar contexto relevante y específico
- Pedir que indique cuando no está seguro
- Solicitar razonamiento paso a paso para tareas complejas
- Especificar formato, restricciones y fuentes esperadas

### RAG (Retrieval-Augmented Generation)

Para combatir conocimiento desactualizado y alucinaciones:

1. Buscar documentos relevantes en una base de datos actualizada
2. Incluirlos como contexto en el prompt
3. El modelo responde basándose en información verificada

> RAG reduce alucinaciones al anclar las respuestas en documentos reales, pero no las elimina por completo.

### Herramientas externas

Conectar el modelo con sistemas especializados:
- **Calculadoras** para operaciones matemáticas precisas
- **APIs de búsqueda** para información en tiempo real
- **Intérpretes de código** para ejecutar y validar programas
- **Bases de datos** para consultas estructuradas

### Seguridad y privacidad

**Vulnerabilidades comunes:**

| Ataque | Qué hace | Mitigación |
|--------|----------|-----------|
| **Prompt Injection** | Instrucciones ocultas que el modelo ejecuta | Validar y sanitizar entradas |
| **Jailbreaking** | Manipular para ignorar restricciones | Filtros de contenido en entrada/salida |
| **Fuga de datos** | Revelar información del entrenamiento | Limitar datos sensibles en prompts |

**Protección de datos:**
- **Anonimizar** información sensible antes de enviarla
- **Modelos locales** para datos confidenciales
- **Verificar políticas** del proveedor sobre retención y uso de datos
- **Monitorear** accesos y patrones de uso sospechosos

> La elección entre modelos en la nube (más capaces) y locales (más privados) depende del equilibrio entre funcionalidad y control de datos.

## Resumen

```mermaid
%%{init: {'theme': 'default'}}%%
flowchart TB
    subgraph origen["Raíz del problema"]
        R["Predicción estadística<br/>sin verificación"]
    end

    subgraph limitaciones["Limitaciones principales"]
        L1["Alucinaciones"]
        L2["Conocimiento desactualizado"]
        L3["Ventana de contexto limitada"]
        L4["Razonamiento débil"]
        L5["Sesgos heredados"]
    end

    subgraph mitigaciones["Estrategias de mitigación"]
        M1["Verificación humana"]
        M2["RAG + Herramientas"]
        M3["Prompting efectivo"]
        M4["Seguridad y privacidad"]
    end

    R --> limitaciones
    limitaciones --> mitigaciones

    style origen fill:#9f7aea,color:#fff
    style limitaciones fill:#ff6b6b,color:#fff
    style mitigaciones fill:#4ade80,color:#fff
```

> Los LLM son herramientas útiles pero imperfectas. Conocer sus limitaciones es esencial para usarlos de forma efectiva y responsable en producción.